In [14]:
import cv2
import os
from collections import defaultdict

# Input and output paths
input_dir = r"../downloaded_videos"
output_dir = r"standardized_videos"

# Check if input directory exists
if not os.path.exists(input_dir):
    print(f"❌ Input directory not found: {os.path.abspath(input_dir)}")
    exit()

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Initialize variables for lowest resolution and FPS
min_width, min_height, min_fps = float("inf"), float("inf"), float("inf")

# Step 1: Analyze all videos to find the lowest resolution and FPS
for filename in os.listdir(input_dir):
    if filename.endswith(".mp4"):
        input_path = os.path.join(input_dir, filename)
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"❌ Error opening {filename}")
            continue
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS)

        # Update the minimum values
        min_width = min(min_width, width)
        min_height = min(min_height, height)
        min_fps = min(min_fps, fps)

        cap.release()

# If no valid videos were found
if min_width == float("inf") or min_height == float("inf") or min_fps == float("inf"):
    print("❌ No valid videos found for analysis.")
    exit()

print(f"Lowest Resolution: {min_width}x{min_height}, Lowest FPS: {int(min_fps)}")
# Step 2: Group videos into clusters based on a pattern (e.g., folder name or prefix)
clusters = defaultdict(list)
for filename in os.listdir(input_dir):
    if filename.endswith(".mp4"):
        cluster_key = filename.split("_")[1]  # Group by prefix (you can customize this pattern)
        clusters[cluster_key].append(filename)
# Step 3: Standardize videos, taking only 50 videos from each cluster
for cluster_key, filenames in clusters.items():
    print(f"Processing cluster: {cluster_key}")
    # Ensure the cluster contains videos
    if len(filenames) == 0:
        print(f"⚠️ No videos found in cluster: {cluster_key}")
        continue
    # Always take the first 50 videos from each cluster (or less if there aren't 50 videos)
    filenames_to_process = filenames[:50]
    for filename in filenames_to_process:
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        # Print the original size
        original_size = os.path.getsize(input_path) / (1024 * 1024)  # Convert to MB
        print(f"📁 {filename} - Original Size: {original_size:.2f} MB")
        # Open video file
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"❌ Error opening {filename}")
            continue
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")

        out = cv2.VideoWriter(output_path, fourcc,min_fps, (min_width, min_height))
        print(f"🔄 Standardizing {filename}...")
        while True:
            ret, frame = cap.read()
            if not ret:
                break  # End of video
            # Resize frame to lowest resolution
            frame = cv2.resize(frame, (min_width, min_height))
            # Write standardized frame
            out.write(frame)
        # Release resources
        cap.release()
        out.release()
        # Print the standardized size
        standardized_size = os.path.getsize(output_path) / (1024 * 1024)  # Convert to MB
        print(f"✅ Standardized: {filename} - New Size: {standardized_size:.2f} MB")
        print(f"📉 Size Reduction: {original_size - standardized_size:.2f} MB\n")
print("🎉 Standardization complete for all videos!")

Lowest Resolution: 420x316, Lowest FPS: 23
Processing cluster: 0
📁 cluster_0_video_1007789530.mp4 - Original Size: 0.98 MB
🔄 Standardizing cluster_0_video_1007789530.mp4...
✅ Standardized: cluster_0_video_1007789530.mp4 - New Size: 3.22 MB
📉 Size Reduction: -2.24 MB

📁 cluster_0_video_1008025297.mp4 - Original Size: 1.56 MB
🔄 Standardizing cluster_0_video_1008025297.mp4...
✅ Standardized: cluster_0_video_1008025297.mp4 - New Size: 5.29 MB
📉 Size Reduction: -3.73 MB

📁 cluster_0_video_1008225385.mp4 - Original Size: 1.17 MB
🔄 Standardizing cluster_0_video_1008225385.mp4...
✅ Standardized: cluster_0_video_1008225385.mp4 - New Size: 3.80 MB
📉 Size Reduction: -2.63 MB

📁 cluster_0_video_1008352831.mp4 - Original Size: 3.80 MB
🔄 Standardizing cluster_0_video_1008352831.mp4...
✅ Standardized: cluster_0_video_1008352831.mp4 - New Size: 3.83 MB
📉 Size Reduction: -0.03 MB

📁 cluster_0_video_1008400516.mp4 - Original Size: 1.09 MB
🔄 Standardizing cluster_0_video_1008400516.mp4...
✅ Standardized: